# CS334 Lab 4: Character-Level RNN Classification (100 pts)

In this lab, you will build a character-level RNN that predicts the language origin of a surname.

Learning goals:
- implement an `nn.RNN` classifier
- train the model with minibatches and negative log-likelihood loss
- evaluate the model with accuracy, top-k predictions, and a confusion matrix

Recommended reading materials: chapter 13 and 14

## Lab Overview

In this lab, you will build a character-level recurrent neural network that reads one surname at a time and predicts its language of origin.

The task:
- input: a surname such as `"Nakamura"` or `"Garcia"`
- output: a predicted language label such as `Japanese`, `Spanish`, or `Italian`

The dataset:
- it contains text files of surnames grouped by language
- each file stores one surname per line
- there are 18 language categories in total

The story behind the dataset:
- surnames often contain spelling patterns that are more common in some languages than others
- a character-level RNN can learn those patterns directly from letters, without using word embeddings
- this is a toy NLP classification problem, so the goal is to learn sequence modeling, not to make claims about a person's identity


## Setup

Run the next two code cells first.

If automatic download is blocked, manually place the dataset in your and rerun.


In [ ]:
import random
import string
import time
import unicodedata
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, Subset, random_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
DATA_URL = "https://download.pytorch.org/tutorial/data.zip"
DATA_ROOT = Path("data")
NAMES_DIR = DATA_ROOT / "names"
ZIP_PATH = Path("data.zip")

if not NAMES_DIR.exists():
    print("Downloading tutorial dataset...")
    urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(".")

txt_files = sorted(NAMES_DIR.glob("*.txt"))
print(f"Found {len(txt_files)} language files in {NAMES_DIR.resolve()}")
print("First five files:", [path.name for path in txt_files[:5]])


## Provided Setup: Data Cleaning and Dataset Construction

These cells are provided so students can focus on the model, training, and evaluation logic instead of text-cleaning and dataset engineering.


In [ ]:
allowed_characters = string.ascii_letters + " .,;'" + "_"
n_letters = len(allowed_characters)

def unicode_to_ascii(s):
    """Convert a Unicode string into plain ASCII characters used by this lab."""
    return "".join(
        c
        for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn" and c in allowed_characters
    )

def letter_to_index(letter):
    """Return the index of one character in allowed_characters."""
    if letter not in allowed_characters:
        return allowed_characters.index("_")
    return allowed_characters.index(letter)

def line_to_tensor(line):
    """Turn one name into a tensor of shape (line_length, 1, n_letters)."""
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li, 0, letter_to_index(letter)] = 1.0
    return tensor


In [ ]:
print("ASCII example:", unicode_to_ascii("Ślusàrski"))
print("index of A:", letter_to_index("A"))
print("index of unknown symbol:", letter_to_index("🙂"))

sample_tensor = line_to_tensor("Ahn")
print("tensor shape:", sample_tensor.shape)
print("sum of first one-hot row:", sample_tensor[0, 0].sum().item())

assert unicode_to_ascii("Ślusàrski") == "Slusarski"
assert line_to_tensor("A").shape == (1, 1, n_letters)
assert torch.isclose(line_to_tensor("A")[0, 0].sum(), torch.tensor(1.0))


In [ ]:
class NamesDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.data = []
        self.data_tensors = []
        self.labels = []
        self.labels_tensors = []

        labels_set = set()
        for filename in sorted(self.data_dir.glob("*.txt")):
            label = filename.stem
            labels_set.add(label)
            raw_lines = filename.read_text(encoding="utf-8").strip().split("\n")
            for name in raw_lines:
                cleaned_name = unicode_to_ascii(name)
                self.data.append(cleaned_name)
                self.data_tensors.append(line_to_tensor(cleaned_name))
                self.labels.append(label)

        self.labels_uniq = sorted(labels_set)
        self.label_to_index = {label: i for i, label in enumerate(self.labels_uniq)}
        for label in self.labels:
            label_tensor = torch.tensor([self.label_to_index[label]], dtype=torch.long)
            self.labels_tensors.append(label_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (
            self.labels_tensors[idx],
            self.data_tensors[idx],
            self.labels[idx],
            self.data[idx],
        )


In [ ]:
alldata = NamesDataset(NAMES_DIR)
train_size = int(0.85 * len(alldata))
test_size = len(alldata) - train_size
train_set, test_set = random_split(
    alldata,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(SEED),
)

label_tensor, name_tensor, label, name = alldata[0]
print(f"loaded {len(alldata)} names across {len(alldata.labels_uniq)} languages")
print("example label:", label)
print("example name:", name)
print("name tensor shape:", name_tensor.shape)

assert len(alldata) > 10000
assert len(alldata.labels_uniq) == 18
assert name_tensor.shape[2] == n_letters


## Task 1: Build the RNN Classifier [20 pts]

Your first job is to complete the model code. The RNN will read a full name character by character, then use the final hidden state to predict the class label.

### Quesions: 
1) [5 pts] explain the CharRNN class based on your understanding
2) [10 pts] complete `forward`.
3) [5 pts] For this classification task, why do we use the final hidden state to make the prediction instead of classifying every character separately?

Expected input/output:
- `forward(line_tensor)` takes a tensor with shape `(line_length, 1, n_letters)`.
- `forward(...)` returns log-probabilities with shape `(1, num_languages)`.

Expected checks after you finish:
- `sample_output.shape` should be `(1, len(alldata.labels_uniq))`
- `sample_output.exp().sum()` should be very close to `1.0`

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        # TODO: complete forward function
        return output

def label_from_output(output, output_labels):
    top_n, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i], label_i

n_hidden = 64
rnn = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq)).to(DEVICE)


In [ ]:
sample_output = rnn(line_to_tensor("Albert"))
guess_label, guess_index = label_from_output(sample_output, alldata.labels_uniq)

print("output shape:", sample_output.shape)
print("probability sum:", sample_output.exp().sum().item())
print("predicted label:", guess_label, guess_index)

assert sample_output.shape == (1, len(alldata.labels_uniq))
assert abs(sample_output.exp().sum().item() - 1.0) < 1e-5


## Task2: Training Loop [20 pts]

Now train the classifier on minibatches of names.

### Questions:
1) [10 pts] complete `train_model`
2) [5 pts] Why do we call `optimizer.zero_grad()` during training? 
3) [5 pts] Why is gradient clipping useful in RNNs?

Expected input/output:
- `train_model(...) -> list[float]`
  Output: one average training loss per epoch.

Expected checks after you finish:
- the debug run should produce a loss list of length `1`

Training workflow:
- switch the model to training mode
- shuffle the training examples each epoch
- accumulate the NLL loss across a minibatch
- call `backward()` once per batch
- clip gradients before the optimizer step
- store one average batch loss per epoch


In [ ]:
def train_model(
    model,
    training_data,
    n_epoch=5,
    n_batch_size=64,
    report_every=1,
    learning_rate=0.15,
):
    criterion = nn.NLLLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    all_losses = []

    model.train()
    for epoch in range(1, n_epoch + 1):
        current_loss = 0.0
        indices = list(range(len(training_data)))
        random.shuffle(indices)
        n_splits = max(1, len(indices) // n_batch_size)
        batches = np.array_split(indices, n_splits)

        for batch in batches:
            batch_loss = 0.0
            for i in batch:
                label_tensor, text_tensor, _, _ = training_data[i]
                label_tensor = label_tensor.to(DEVICE)
                text_tensor = text_tensor.to(DEVICE)
                # TODO: compute output
                # TODO: compute loss

            # TODO: loss back propagation
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
            optimizer.zero_grad()
            current_loss += batch_loss.item() / len(batch)

        epoch_loss = current_loss / len(batches)
        all_losses.append(epoch_loss)
        if epoch % report_every == 0:
            print(f"epoch {epoch:02d}/{n_epoch}: avg batch loss = {epoch_loss:.4f}")

    return all_losses


In [ ]:
debug_train = Subset(train_set, range(min(256, len(train_set))))
debug_model = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq)).to(DEVICE)

debug_losses = train_model(
    debug_model,
    debug_train,
    n_epoch=1,
    n_batch_size=32,
    report_every=1,
    learning_rate=0.15,
)

print("debug losses:", debug_losses)
assert len(debug_losses) == 1

## Task 3: Evaluation [25 pts]

Quetions:

1) [10 pts] complete `evaluate_model`
2) [10 pts] complete `predict`
3) [5 pts] What does the confusion matrix tell you that the single accuracy number does not?

Expected input/output:
- `evaluate_model(...) -> (accuracy, confusion)`
  Output: a scalar accuracy and a normalized confusion matrix with shape `(num_languages, num_languages)`.
- `predict(model, name, classes, n=3) -> list[tuple[str, float]]`
  Output: the top-`n` predicted labels and their log-probabilities.

Expected checks after you finish:
- the confusion matrix should have one row and one column per language
- `predict(..., n=3)` should return exactly three guesses


In [ ]:
def evaluate_model(model, testing_data, classes):
    confusion = torch.zeros(len(classes), len(classes))
    correct = 0

    # TODO: turn on eval mode
    with torch.no_grad():
        for i in range(len(testing_data)):
            label_tensor, text_tensor, label, _ = testing_data[i]
            # TODO: compute output
            _, guess_i = label_from_output(output.cpu(), classes)
            label_i = classes.index(label)
            confusion[label_i, guess_i] += 1
            correct += int(label_i == guess_i)

    for i in range(len(classes)):
        row_sum = confusion[i].sum()
        if row_sum > 0:
            confusion[i] = confusion[i] / row_sum

    accuracy = correct / len(testing_data)
    return accuracy, confusion

def predict(model, name, classes, n=3):
    clean_name = unicode_to_ascii(name)
    with torch.no_grad():
        # TODO: turn on eval mode
        # TODO: top_values, top_indices using topk function

    predictions = []
    for log_prob, idx in zip(top_values[0].cpu(), top_indices[0].cpu()):
        predictions.append((classes[idx.item()], float(log_prob.item())))
    return predictions


In [ ]:
debug_test = Subset(test_set, range(min(128, len(test_set))))
debug_accuracy, debug_confusion = evaluate_model(debug_model, debug_test, alldata.labels_uniq)
debug_predictions = predict(debug_model, "Satoshi", alldata.labels_uniq, n=3)

print("debug accuracy:", debug_accuracy)
print("debug predictions:", debug_predictions)

assert debug_confusion.shape == (len(alldata.labels_uniq), len(alldata.labels_uniq))
assert len(debug_predictions) == 3

## Task 4: Results Analysis (15 pts)

After your debug checks pass, run the next two cells to train on the full training split and inspect the results.

Write short answers after you run the full model:

1. [5 pts] What does the training-loss curve suggest about learning progress across epochs?
2. [5 pts] Which language pairs seem to be confused most often in the confusion matrix?
3. [5 pts] Look at the top-3 predictions for `"Satoshi"` and `"Garcia"`. Do the guesses make sense? Why or why not?


In [ ]:
torch.manual_seed(SEED)
full_model = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq)).to(DEVICE)

start = time.time()
loss_history = train_model(
    full_model,
    train_set,
    n_epoch=5,
    n_batch_size=64,
    report_every=1,
    learning_rate=0.15,
)
elapsed = time.time() - start

test_accuracy, confusion = evaluate_model(full_model, test_set, alldata.labels_uniq)
print(f"training time: {elapsed:.1f} seconds")
print(f"test accuracy: {test_accuracy:.3f}")
print("Top-3 predictions for 'Satoshi':", predict(full_model, "Satoshi", alldata.labels_uniq))
print("Top-3 predictions for 'Garcia':", predict(full_model, "Garcia", alldata.labels_uniq))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_history)
axes[0].set_title("Training Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Average batch loss")

im = axes[1].imshow(confusion.numpy(), cmap="Blues")
axes[1].set_title("Normalized Confusion Matrix")
axes[1].set_xticks(range(len(alldata.labels_uniq)))
axes[1].set_xticklabels(alldata.labels_uniq, rotation=90)
axes[1].set_yticks(range(len(alldata.labels_uniq)))
axes[1].set_yticklabels(alldata.labels_uniq)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## Task 5: Extra Questions (20 pts)

1. [5 pts] Why is a character-level model a reasonable choice for surname classification?
2. [5 pts] What information might this model miss because it only uses spelling?
3. [5 pts] If you replaced RNN with LSTM`, what change would you expect in performance or training stability?
4. [5 pts] Why is accuracy alone not enough to understand this model's behavior?
